# DEATHMAP AI
### Mortality Pattern Discovery & Healthcare Accessibility Intelligence

A single-cell Python research prototype for synthetic mortality trend analysis, healthcare-access indicators, outlier detection, feature importance and screening priority exploration.

**Important:** All data are artificially generated. This notebook contains no real Bengaluru Rural or India mortality statistics. The screening index is illustrative, not validated. Associations do not establish causation or preventability.

In [ ]:
# DEATHMAP AI | Single-cell mortality pattern research prototype
# IMPORTANT: Entirely synthetic illustrative data; not real mortality statistics.

import sys, subprocess, importlib.util, warnings
warnings.filterwarnings("ignore")

for module, package in {
    "numpy":"numpy", "pandas":"pandas", "matplotlib":"matplotlib",
    "seaborn":"seaborn", "sklearn":"scikit-learn", "plotly":"plotly"
}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

# 1. Generate synthetic community-level data
rng = np.random.default_rng(42)
n = 1200
population = rng.integers(1800, 18000, n)
elderly = np.clip(rng.normal(11, 4, n), 2, 28)
poverty = np.clip(rng.normal(24, 12, n), 2, 70)
travel = np.clip(rng.gamma(2.4, 13, n), 2, 110)
facilities = np.clip(rng.normal(1.8, .8, n), .1, 5)
water = np.clip(rng.normal(78, 14, n), 25, 100)
sanitation = np.clip(rng.normal(71, 17, n), 15, 100)
care = np.clip(100 - travel*.38 + rng.normal(0, 9, n), 8, 100)
year = rng.integers(2019, 2026, n)

rate = np.clip(
    360 + elderly*9 + poverty*1.5 + travel*1.15 - facilities*22
    - water*.65 - sanitation*.35 - care*.25
    + (year-2019)*1.8 + rng.normal(0, 48, n), 180, 1100
)
deaths = np.maximum(0, np.round(population*rate/100000 + rng.normal(0, 1.2, n))).astype(int)

df = pd.DataFrame({
    "Community_ID":[f"COMM_{i:04d}" for i in range(1,n+1)],
    "Year":year, "Population":population, "Elderly_Percent":elderly,
    "Poverty_Percent":poverty, "Travel_Minutes_To_Care":travel,
    "Facilities_Per_10k":facilities, "Water_Access_Percent":water,
    "Sanitation_Percent":sanitation, "Primary_Care_Coverage":care,
    "Death_Count":deaths, "Mortality_Rate_Per_100k":rate
})

# Add missingness and clean
for col in ["Elderly_Percent","Poverty_Percent","Travel_Minutes_To_Care","Water_Access_Percent"]:
    idx = rng.choice(df.index, size=int(.015*n), replace=False)
    df.loc[idx,col] = np.nan
num = df.select_dtypes(include=np.number).columns
df[num] = df[num].fillna(df[num].median())

print("="*68, "\\nDEATHMAP AI | MORTALITY & ACCESS INTELLIGENCE\\n", "="*68)
print("Dataset shape:", df.shape, "| Missing values after cleaning:", df.isna().sum().sum())
display(df.head())

# 2. Regression model: estimate synthetic mortality rate
features = [
    "Elderly_Percent","Poverty_Percent","Travel_Minutes_To_Care",
    "Facilities_Per_10k","Water_Access_Percent","Sanitation_Percent",
    "Primary_Care_Coverage","Population","Year"
]
X, y = df[features], df["Mortality_Rate_Per_100k"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.25, random_state=42
)
model = RandomForestRegressor(
    n_estimators=250, max_depth=12, min_samples_leaf=3,
    random_state=42, n_jobs=-1
)
model.fit(X_train,y_train)
pred = model.predict(X_test)
mae = mean_absolute_error(y_test,pred)
rmse = np.sqrt(mean_squared_error(y_test,pred))
r2 = r2_score(y_test,pred)
print("\\nSYNTHETIC MODEL EVALUATION")
print(f"MAE: {mae:.2f} | RMSE: {rmse:.2f} | R²: {r2:.3f}")

# 3. Explainability and outlier detection
importance = pd.DataFrame({
    "Feature":features, "Importance":model.feature_importances_
}).sort_values("Importance",ascending=False)
perm = permutation_importance(
    model,X_test,y_test,n_repeats=5,random_state=42,
    scoring="neg_mean_absolute_error"
)
permutation_df = pd.DataFrame({
    "Feature":features,"Permutation_Importance":perm.importances_mean
}).sort_values("Permutation_Importance",ascending=False)

outlier_model = IsolationForest(
    n_estimators=200, contamination=.05, random_state=42
)
outlier_cols = [
    "Mortality_Rate_Per_100k","Travel_Minutes_To_Care",
    "Facilities_Per_10k","Poverty_Percent","Elderly_Percent"
]
df["Statistical_Outlier_Flag"] = (
    outlier_model.fit_predict(df[outlier_cols]) == -1
).astype(int)

# 4. Transparent screening index (not a validated risk score)
def norm(s):
    span = s.max()-s.min()
    return (s-s.min())/span if span else s*0

df["Access_Barrier_Index"] = (
    .35*norm(df["Travel_Minutes_To_Care"])
    + .25*(1-norm(df["Facilities_Per_10k"]))
    + .20*(1-norm(df["Primary_Care_Coverage"]))
    + .20*norm(df["Poverty_Percent"])
)
z = (df["Mortality_Rate_Per_100k"]-df["Mortality_Rate_Per_100k"].mean())/df["Mortality_Rate_Per_100k"].std()
df["Screening_Priority_Index"] = .55*norm(z.clip(lower=0)) + .45*df["Access_Barrier_Index"]
df["Screening_Category"] = pd.cut(
    df["Screening_Priority_Index"],[-.001,.33,.66,1.001],
    labels=["Lower","Moderate","Higher"]
)

print("\\nTOP FEATURES")
display(importance)
print("\\nPERMUTATION IMPORTANCE")
display(permutation_df)
print("\\nSCREENING CATEGORY COUNTS")
display(df["Screening_Category"].value_counts().rename_axis("Category").to_frame("Communities"))

# 5. Yearly summary
yearly = df.groupby("Year",as_index=False).agg(
    Mean_Mortality_Rate=("Mortality_Rate_Per_100k","mean"),
    Synthetic_Deaths=("Death_Count","sum"),
    Communities=("Community_ID","count")
)
display(yearly.round(2))

# 6. Dashboard
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(2,2,figsize=(16,10))
sns.histplot(df["Mortality_Rate_Per_100k"],kde=True,ax=ax[0,0])
ax[0,0].set_title("Synthetic Mortality Rate Distribution")
sns.barplot(data=importance.head(8),x="Importance",y="Feature",ax=ax[0,1])
ax[0,1].set_title("Random Forest Feature Importance")
sns.lineplot(data=yearly,x="Year",y="Mean_Mortality_Rate",marker="o",ax=ax[1,0])
ax[1,0].set_title("Yearly Synthetic Mortality Trend")
sns.countplot(data=df,x="Screening_Category",hue="Screening_Category",legend=False,ax=ax[1,1])
ax[1,1].set_title("Communities by Screening Category")
plt.suptitle("DEATHMAP AI | Research Analytics Dashboard",fontsize=16,fontweight="bold")
plt.tight_layout()
plt.show()

# 7. Interactive exploratory plots
fig = px.scatter(
    df.sample(700,random_state=42),
    x="Travel_Minutes_To_Care",y="Mortality_Rate_Per_100k",
    color="Screening_Category",size="Population",
    hover_data=["Community_ID","Year","Poverty_Percent","Facilities_Per_10k"],
    title="Healthcare Access vs Synthetic Mortality Rate",
    template="plotly_white"
)
fig.show()
px.line(
    yearly,x="Year",y="Mean_Mortality_Rate",markers=True,
    title="Yearly Synthetic Mortality Trend",template="plotly_white"
).show()

# 8. Investigation shortlist and exports
shortlist = df.nlargest(15,"Screening_Priority_Index")[
    ["Community_ID","Year","Mortality_Rate_Per_100k",
     "Travel_Minutes_To_Care","Facilities_Per_10k",
     "Primary_Care_Coverage","Access_Barrier_Index",
     "Screening_Priority_Index","Statistical_Outlier_Flag"]
]
print("\\nTOP 15 SYNTHETIC RECORDS FOR FURTHER REVIEW")
display(shortlist.round(2))

df.to_csv("deathmap_ai_synthetic_dataset.csv",index=False)
importance.to_csv("deathmap_ai_feature_importance.csv",index=False)
shortlist.to_csv("deathmap_ai_investigation_shortlist.csv",index=False)
yearly.to_csv("deathmap_ai_yearly_summary.csv",index=False)

print("\\nExported 4 CSV files.")
print("DEATHMAP AI EXECUTION COMPLETED.")
print("\\nLIMITATIONS: All records are synthetic. The screening index is illustrative,")
print("not clinically validated. Statistical associations do not prove causation,")
print("preventability, or actual mortality patterns in any location.")
